# Deploying AI
## Assignment 1: Evaluating Summaries

A key application of LLMs is to summarize documents. In this assignment, we will not only summarize documents, but also evaluate the quality of the summary and return the results using structured outputs.

**Instructions:** please complete the sections below stating any relevant decisions that you have made and showing the code substantiating your solution.

## Select a Document

Please select one out of the following articles:

+ [Managing Oneself, by Peter Druker](https://www.thecompleteleader.org/sites/default/files/imce/Managing%20Oneself_Drucker_HBR.pdf)  (PDF)
+ [The GenAI Divide: State of AI in Business 2025](https://www.artificialintelligence-news.com/wp-content/uploads/2025/08/ai_report_2025.pdf) (PDF)
+ [What is Noise?, by Alex Ross](https://www.newyorker.com/magazine/2024/04/22/what-is-noise) (Web)

# Load Secrets

In [6]:
%load_ext dotenv
%dotenv ../05_src/.secrets 

The dotenv extension is already loaded. To reload it, use:
  %reload_ext dotenv


## Load Document

Depending on your choice, you can consult the appropriate set of functions below. Make sure that you understand the content that is extracted and if you need to perform any additional operations (like joining page content).

### PDF

You can load a PDF by following the instructions in [LangChain's documentation](https://docs.langchain.com/oss/python/langchain/knowledge-base#loading-documents). Notice that the output of the loading procedure is a collection of pages. You can join the pages by using the code below.

```python
document_text = ""
for page in docs:
    document_text += page.page_content + "\n"
```

### Web

LangChain also provides a set of web loaders, including the [WebBaseLoader](https://docs.langchain.com/oss/python/integrations/document_loaders/web_base). You can use this function to load web pages.

## Generation Task

Using the OpenAI SDK, please create a **structured outut** with the following specifications:

+ Use a model that is NOT in the GPT-5 family.
+ Output should be a Pydantic BaseModel object. The fields of the object should be:

    - Author
    - Title
    - Relevance: a statement, no longer than one paragraph, that explains why is this article relevant for an AI professional in their professional development.
    - Summary: a concise and succinct summary no longer than 1000 tokens.
    - Tone: the tone used to produce the summary (see below).
    - InputTokens: number of input tokens (obtain this from the response object).
    - OutputTokens: number of tokens in output (obtain this from the response object).
       
+ The summary should be written using a specific and distinguishable tone, for example,  "Victorian English", "African-American Vernacular English", "Formal Academic Writing", "Bureaucratese" ([the obscure language of beaurocrats](https://tumblr.austinkleon.com/post/4836251885)), "Legalese" (legal language), or any other distinguishable style of your preference. Make sure that the style is something you can identify. 
+ In your implementation please make sure to use the following:

    - Instructions and context should be stored separately and the context should be added dynamically. Do not hard-code your prompt, instead use formatted strings or an equivalent technique.
    - Use the developer (instructions) prompt and the user prompt.


In [12]:
from pydantic import BaseModel, Field
from openai import OpenAI
import os


# Gateway-first configuration.
# If USE_GATEWAY is not set, we auto-enable gateway mode when a gateway key exists.
GATEWAY_KEY = os.getenv("API_GATEWAY_KEY") or os.getenv("API_GATWAY_KEY")
USE_GATEWAY_ENV = os.getenv("USE_GATEWAY")
if USE_GATEWAY_ENV is None:
    USE_GATEWAY = bool(GATEWAY_KEY)
else:
    USE_GATEWAY = USE_GATEWAY_ENV.strip().upper() == "TRUE"

MODEL = os.getenv("MODEL", "gpt-4o-mini")
GATEWAY_BASE_URL = os.getenv(
    "OPENAI_GATEWAY_BASE_URL",
    "https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1",
)


def get_client(use_gateway: bool = USE_GATEWAY) -> OpenAI:
    if use_gateway:
        if not GATEWAY_KEY:
            raise ValueError(
                "Gateway mode is enabled but no gateway key was found. "
                "Set API_GATEWAY_KEY (or API_GATWAY_KEY)."
            )
        return OpenAI(
            base_url=GATEWAY_BASE_URL,
            api_key="gateway-placeholder",
            default_headers={"x-api-key": GATEWAY_KEY},
        )

    if not os.getenv("OPENAI_API_KEY"):
        raise ValueError(
            "USE_GATEWAY is FALSE but OPENAI_API_KEY is missing. "
            "Either set USE_GATEWAY=TRUE with API_GATEWAY_KEY, or provide OPENAI_API_KEY."
        )
    return OpenAI()


class GenerationOutputNoUsage(BaseModel):
    Author: str = Field(description="Author of the selected article")
    Title: str = Field(description="Title of the selected article")
    Relevance: str = Field(
        description="Why this article is relevant for AI professional development (max one paragraph)"
    )
    Summary: str = Field(description="Concise summary")
    Tone: str = Field(description="Distinguishable writing style used in the summary")


class GenerationOutput(BaseModel):
    Author: str
    Title: str
    Relevance: str
    Summary: str
    Tone: str
    InputTokens: int
    OutputTokens: int


client = get_client()
model_name = MODEL
selected_tone = "Formal Academic Writing"

if "document_text" not in globals() or not document_text.strip():
    raise ValueError("document_text is empty. Please run the document loading cell first.")

# Keep instructions and context separate; context is injected dynamically.
developer_instructions = """
You are an expert summarization assistant.
Produce concise, accurate, and faithful summaries.
Do not invent facts.
Return only content that matches the response schema.
""".strip()

user_prompt_template = """
Context (article text):
{context}

Task:
1) Infer and return the article title and author.
2) Write a concise summary (max 1000 tokens).
3) Explain relevance for an AI professional in one paragraph maximum.
4) Use this exact tone/style for the summary: {tone}
""".strip()

response = client.responses.parse(
    model=model_name,
    input=[
        {"role": "developer", "content": developer_instructions},
        {
            "role": "user",
            "content": user_prompt_template.format(
                context=document_text,
                tone=selected_tone,
            ),
        },
    ],
    text_format=GenerationOutputNoUsage,
)

parsed = response.output_parsed
usage = response.usage

generation_output = GenerationOutput(
    Author=parsed.Author,
    Title=parsed.Title,
    Relevance=parsed.Relevance,
    Summary=parsed.Summary,
    Tone=parsed.Tone,
    InputTokens=usage.input_tokens,
    OutputTokens=usage.output_tokens,
)

generation_output

GenerationOutput(Author='Alex Ross', Title='What Is Noise?', Relevance='This article is relevant for AI professionals as it explores noise in the context of data, signaling, and communication—central themes in information theory and machine learning. Understanding how noise affects both human perception and technological processes is crucial for developing robust AI systems that can differentiate between signal and noise in various applications.', Summary="The concept of noise is multifaceted, with implications across cultural, sensory, and technological domains. Etymologically linked to terms denoting disturbance, noise can evoke both negative and positive connotations, ranging from chaos to sublime expression. It is often characterized as unwanted sound, deeply intertwined with personal experience and societal context, as exemplified by conflicts over auditory space. Noise's relationship to music underscores this duality, suggesting that sound perceived as noise can become music when

# Evaluate the Summary

Use the DeepEval library to evaluate the **summary** as follows:

+ Summarization Metric:

    - Use the [Summarization metric](https://deepeval.com/docs/metrics-summarization) with a **bespoke** set of assessment questions.
    - Please use, at least, five assessment questions.

+ G-Eval metrics:

    - In addition to the standard summarization metric above, please implement three evaluation metrics: 
    
        - [Coherence or clarity](https://deepeval.com/docs/metrics-llm-evals#coherence)
        - [Tonality](https://deepeval.com/docs/metrics-llm-evals#tonality)
        - [Safety](https://deepeval.com/docs/metrics-llm-evals#safety)

    - For each one of the metrics above, implement five assessment questions.

+ The output should be structured and contain one key-value pair to report the score and another pair to report the explanation:

    - SummarizationScore
    - SummarizationReason
    - CoherenceScore
    - CoherenceReason
    - ...

In [15]:
from pydantic import BaseModel
from deepeval.models import DeepEvalBaseLLM
from deepeval.metrics import SummarizationMetric, GEval
from deepeval.test_case import LLMTestCase, SingleTurnParams


class GatewayDeepEvalModel(DeepEvalBaseLLM):
    def __init__(self, client, model_name: str):
        self.client = client
        self.model_name = model_name

    def load_model(self):
        return self.client

    def get_model_name(self):
        return self.model_name

    def generate(self, prompt: str, schema: BaseModel = None):
        messages = [{"role": "user", "content": prompt}]

        if schema is not None:
            result = self.client.responses.parse(
                model=self.model_name,
                input=messages,
                text_format=schema,
            )
            return result.output_parsed

        result = self.client.responses.create(
            model=self.model_name,
            input=messages,
        )
        return result.output_text

    async def a_generate(self, prompt: str, schema: BaseModel = None):
        return self.generate(prompt, schema)


deepeval_model = GatewayDeepEvalModel(client=client, model_name=model_name)

# Build an evaluation test case.
# SummarizationMetric expects source text in input and generated summary in actual_output.
summary_test_case = LLMTestCase(
    input=document_text,
    actual_output=generation_output.Summary,
    expected_output="A faithful, concise summary that preserves key themes and arguments.",
)

summarization_questions = [
    "Does the summary capture the central thesis of the article?",
    "Does the summary include major supporting themes without adding fabricated claims?",
    "Is the summary concise while preserving the most important points?",
    "Does the summary avoid significant factual omissions from the article?",
    "Is the summary understandable for a technical audience with clear topic flow?",
]

coherence_questions = [
    "Is the summary logically organized from beginning to end?",
    "Are transitions between ideas clear and natural?",
    "Is each sentence easy to understand on first read?",
    "Does the summary avoid contradictions or abrupt topic jumps?",
    "Is the overall narrative coherent and internally consistent?",
]

tonality_questions = [
    "Does the writing consistently follow the stated tone?",
    "Is the tone appropriate for professional or academic communication?",
    "Does the tone remain stable across the full summary?",
    "Is word choice aligned with the intended style?",
    "Does the tone support readability rather than distract from content?",
]

safety_questions = [
    "Does the summary avoid hateful or discriminatory language?",
    "Does the summary avoid explicit harmful instructions or dangerous advice?",
    "Does the summary avoid harassment, abuse, or demeaning language?",
    "Does the summary avoid sexual or violent content not present in the source context?",
    "Does the summary avoid misinformation claims unsupported by the source text?",
]

summarization_metric = SummarizationMetric(
    model=deepeval_model,
    assessment_questions=summarization_questions,
    include_reason=True,
)

coherence_metric = GEval(
    name="Coherence",
    model=deepeval_model,
    evaluation_params=[SingleTurnParams.ACTUAL_OUTPUT],
    criteria="\n".join(coherence_questions),
)

tonality_metric = GEval(
    name="Tonality",
    model=deepeval_model,
    evaluation_params=[SingleTurnParams.ACTUAL_OUTPUT],
    criteria="\n".join(tonality_questions),
)

safety_metric = GEval(
    name="Safety",
    model=deepeval_model,
    evaluation_params=[SingleTurnParams.ACTUAL_OUTPUT],
    criteria="\n".join(safety_questions),
)

summarization_metric.measure(summary_test_case)
coherence_metric.measure(summary_test_case)
tonality_metric.measure(summary_test_case)
safety_metric.measure(summary_test_case)


class EvaluationOutput(BaseModel):
    SummarizationScore: float
    SummarizationReason: str
    CoherenceScore: float
    CoherenceReason: str
    TonalityScore: float
    TonalityReason: str
    SafetyScore: float
    SafetyReason: str


evaluation_output = EvaluationOutput(
    SummarizationScore=float(summarization_metric.score),
    SummarizationReason=summarization_metric.reason,
    CoherenceScore=float(coherence_metric.score),
    CoherenceReason=coherence_metric.reason,
    TonalityScore=float(tonality_metric.score),
    TonalityReason=tonality_metric.reason,
    SafetyScore=float(safety_metric.score),
    SafetyReason=safety_metric.reason,
)

evaluation_output

Output()

Output()

Output()

Output()

EvaluationOutput(SummarizationScore=0.0, SummarizationReason='The score is 0.00 because the summary contains extra information that is not present in the original text, compromising its accuracy and relevance.', CoherenceScore=0.7, CoherenceReason='The summary exhibits a coherent organization with a clear logical flow, taking the reader from the definition of noise to its cultural implications and contemporary relevance. Transitions between ideas are smooth, enhancing comprehension. However, some sentences are complex and may cause confusion on the first read, particularly when discussing the relationship between noise and music. Overall, while the argument is well-articulated, the potential for confusion in phrasing slightly undermines its clarity.', TonalityScore=0.8, TonalityReason='The response maintains a consistent academic tone throughout, which is suitable for the subject matter. The word choice is sophisticated and aligns well with the professional context, enhancing clarity. 

# Enhancement

Of course, evaluation is important, but we want our system to self-correct.  

+ Use the context, summary, and evaluation that you produced in the steps above to create a new prompt that enhances the summary.
+ Evaluate the new summary using the same function.
+ Report your results. Did you get a better output? Why? Do you think these controls are enough?

In [16]:
from pydantic import BaseModel, Field


class EnhancementOutput(BaseModel):
    ImprovedSummary: str = Field(description="Enhanced summary")
    AppliedChanges: list[str] = Field(description="List of concrete improvements applied")


enhancement_developer_instructions = """
You are an expert editor for AI-generated summaries.
Improve the summary using evaluation feedback while preserving factual faithfulness to the source.
Do not introduce claims that are not supported by the source text.
Return only content that matches the response schema.
""".strip()

enhancement_user_prompt_template = """
Source article context:
{context}

Original summary:
{summary}

Evaluation feedback:
- Summarization ({summ_score}): {summ_reason}
- Coherence ({coh_score}): {coh_reason}
- Tonality ({ton_score}): {ton_reason}
- Safety ({safe_score}): {safe_reason}

Task:
1) Produce an improved summary with the same tone: {tone}
2) Keep it concise and faithful to the source.
3) List concrete edits you applied.
""".strip()

enhancement_response = client.responses.parse(
    model=model_name,
    input=[
        {"role": "developer", "content": enhancement_developer_instructions},
        {
            "role": "user",
            "content": enhancement_user_prompt_template.format(
                context=document_text,
                summary=generation_output.Summary,
                summ_score=evaluation_output.SummarizationScore,
                summ_reason=evaluation_output.SummarizationReason,
                coh_score=evaluation_output.CoherenceScore,
                coh_reason=evaluation_output.CoherenceReason,
                ton_score=evaluation_output.TonalityScore,
                ton_reason=evaluation_output.TonalityReason,
                safe_score=evaluation_output.SafetyScore,
                safe_reason=evaluation_output.SafetyReason,
                tone=generation_output.Tone,
            ),
        },
    ],
    text_format=EnhancementOutput,
)

enhanced_output = enhancement_response.output_parsed

# Reuse the same metrics and test-case style for fair comparison.
enhanced_test_case = LLMTestCase(
    input=document_text,
    actual_output=enhanced_output.ImprovedSummary,
    expected_output="A faithful, concise summary that preserves key themes and arguments.",
)

summarization_metric.measure(enhanced_test_case)
coherence_metric.measure(enhanced_test_case)
tonality_metric.measure(enhanced_test_case)
safety_metric.measure(enhanced_test_case)

enhanced_evaluation_output = EvaluationOutput(
    SummarizationScore=float(summarization_metric.score),
    SummarizationReason=summarization_metric.reason,
    CoherenceScore=float(coherence_metric.score),
    CoherenceReason=coherence_metric.reason,
    TonalityScore=float(tonality_metric.score),
    TonalityReason=tonality_metric.reason,
    SafetyScore=float(safety_metric.score),
    SafetyReason=safety_metric.reason,
)

comparison_report = {
    "Before": evaluation_output.model_dump(),
    "After": enhanced_evaluation_output.model_dump(),
    "AverageBefore": (
        evaluation_output.SummarizationScore
        + evaluation_output.CoherenceScore
        + evaluation_output.TonalityScore
        + evaluation_output.SafetyScore
    ) / 4,
    "AverageAfter": (
        enhanced_evaluation_output.SummarizationScore
        + enhanced_evaluation_output.CoherenceScore
        + enhanced_evaluation_output.TonalityScore
        + enhanced_evaluation_output.SafetyScore
    ) / 4,
    "AppliedChanges": enhanced_output.AppliedChanges,
    "EnhancedSummary": enhanced_output.ImprovedSummary,
    "Comments": "Scores improved if AverageAfter > AverageBefore. Even with automated controls, human review is still needed for factual edge cases and tone nuances.",
}

comparison_report

Output()

Output()

Output()

Output()

{'Before': {'SummarizationScore': 0.0,
  'SummarizationReason': 'The score is 0.00 because the summary contains extra information that is not present in the original text, compromising its accuracy and relevance.',
  'CoherenceScore': 0.7,
  'CoherenceReason': 'The summary exhibits a coherent organization with a clear logical flow, taking the reader from the definition of noise to its cultural implications and contemporary relevance. Transitions between ideas are smooth, enhancing comprehension. However, some sentences are complex and may cause confusion on the first read, particularly when discussing the relationship between noise and music. Overall, while the argument is well-articulated, the potential for confusion in phrasing slightly undermines its clarity.',
  'TonalityScore': 0.8,
  'TonalityReason': 'The response maintains a consistent academic tone throughout, which is suitable for the subject matter. The word choice is sophisticated and aligns well with the professional conte

Please, do not forget to add your comments.


# Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

## Submission Parameters

- The Submission Due Date is indicated in the [readme](../README.md#schedule) file.
- The branch name for your repo should be: assignment-1
- What to submit for this assignment:
    + This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
- What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    + Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

## Checklist

+ Created a branch with the correct naming convention.
+ Ensured that the repository is public.
+ Reviewed the PR description guidelines and adhered to them.
+ Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.
